In [ ]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

In [ ]:
#Import Crew agent classes
from crewai import Agent, Task, Crew

In [ ]:
# Lets set up our keys and model that we are going to be using
import os
os.environ["CREWAI_TESTING"] = "true"
from utils import get_openai_api_key, pretty_print_result, get_serper_api_key

openai_api_key = get_openai_api_key()
os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'
os.environ["SERPER_API_KEY"] = get_serper_api_key()


In [ ]:
'''
import os
from utils import get_gemini_api_key
from utils import get_serper_api_key

gemini_api_key = get_gemini_api_key()
os.environ["GEMINI_MODEL_NAME"] = "models/gemini-2.5-flash-lite-preview-06-17"
os.environ["SERPER_API_KEY"] =get_serper_api_key()
'''

In [ ]:
# Our Agent 1 is Sales Representative, this agent has one single goal
sales_rep_agent = Agent(
    role="Sales Representative",
    goal="Identify high-value leads that match "
         "our ideal customer profile",
    backstory=(
        "As a part of the dynamic sales team at CrewAI, "
        "your mission is to scour "
        "the digital landscape for potential leads. "
        "Armed with cutting-edge tools "
        "and a strategic mindset, you analyze data, "
        "trends, and interactions to "
        "unearth opportunities that others might overlook. "
        "Your work is crucial in paving the way "
        "for meaningful engagements and driving the company's growth."
    ),
    allow_delegation=False,
    verbose=True
)

In [ ]:
# Our Agent 2 is Lead Sales Representative
lead_sales_rep_agent = Agent(
    role="Lead Sales Representative",
    goal="Nurture leads with personalized, compelling communications",
    backstory=(
        "Within the vibrant ecosystem of CrewAI's sales department, "
        "you stand out as the bridge between potential clients "
        "and the solutions they need."
        "By creating engaging, personalized messages, "
        "you not only inform leads about our offerings "
        "but also make them feel seen and heard."
        "Your role is pivotal in converting interest "
        "into action, guiding leads through the journey "
        "from curiosity to commitment."
    ),
    allow_delegation=False,
)   

In [ ]:
# We go ahead and import our tools, we gonna be using two tools in this case, 
from crewai_tools import DirectoryReadTool, \
                         FileReadTool, \
                         SerperDevTool

In [ ]:
# we create instance of the tools
directory_read_tool = DirectoryReadTool(directory='./instructions')
file_read_tool = FileReadTool()
search_tool = SerperDevTool()

In [ ]:
# so in here you can see that we are importing the langchain base model and creating a class that is going to inherite the vanue details
# which have these attributes, name ---
# the reason why we are creating this is bcos so that our agent can work with it and  populate instance of this as they work
from langchain.tools import BaseTool
from pydantic import BaseModel

class SentimentAnalysisTool(BaseTool):
    name: str = "sentiment_analysis_tool"
    description: str = "Analyzes sentiment to ensure positive and engaging communication."

    def _run(self, text: str):
        # Your sentiment logic here
        return "positive"

    async def _arun(self, text: str):
        raise NotImplementedError("Async not supported")


In [ ]:
from langchain.tools import BaseTool

class SentimentAnalysisTool(BaseTool):
    name: str = "sentiment_analysis_tool"
    description: str = "Analyzes sentiment of text"

    # Implement the synchronous run method
    def _run(self, text: str):
        # Example logic
        if "bad" in text.lower():
            return "negative"
        return "positive"

    # Implement the async run method (can just raise NotImplementedError if not needed)
    async def _arun(self, text: str):
        raise NotImplementedError("Async not implemented")

# Now you can instantiate your subclass
sentiment_analysis_tool = SentimentAnalysisTool()


In [ ]:
# so let go ahead and create our first task. this task is responsable to conduct an in-depth analysis
# so you can see
# that it expect few input here, lead_name and industry and here we are using other attributes 

lead_profiling_task = Task(
    description=(
        "Conduct an in-depth analysis of {lead_name}, "
        "a company in the {industry} sector "
        "that recently showed interest in our solutions. "
        "Utilize all available data sources "
        "to compile a detailed profile, "
        "focusing on key decision-makers, recent business "
        "developments, and potential needs "
        "that align with our offerings. "
        "This task is crucial for tailoring "
        "our engagement strategy effectively.\n"
        "Don't make assumptions and "
        "only use information you absolutely sure about."
    ),
    expected_output=(
        "A comprehensive report on {lead_name}, "
        "including company background, "
        "key personnel, recent milestones, and identified needs. "
        "Highlight potential areas where "
        "our solutions can provide value, "
        "and suggest personalized engagement strategies."
    ),
    tools=[directory_read_tool, file_read_tool, search_tool],
    agent=sales_rep_agent,
)

In [ ]:
# so now lets create ouw second task, this task is responsiablle for Using the insight gathered from
# and its expecting a few diff variables...
personalized_outreach_task = Task(
    description=(
        "Using the insights gathered from "
        "the lead profiling report on {lead_name}, "
        "craft a personalized outreach campaign "
        "aimed at {key_decision_maker}, "
        "the {position} of {lead_name}. "
        "The campaign should address their recent {milestone} "
        "and how our solutions can support their goals. "
        "Your communication must resonate "
        "with {lead_name}'s company culture and values, "
        "demonstrating a deep understanding of "
        "their business and needs.\n"
        "Don't make assumptions and only "
        "use information you absolutely sure about."
    ),
    expected_output=(
        "A series of personalized email drafts "
        "tailored to {lead_name}, "
        "specifically targeting {key_decision_maker}."
        "Each draft should include "
        "a compelling narrative that connects our solutions "
        "with their recent achievements and future goals. "
        "Ensure the tone is engaging, professional, "
        "and aligned with {lead_name}'s corporate identity."
    ),
    tools=[sentiment_analysis_tool, search_tool],
    agent=lead_sales_rep_agent,
)

In [ ]:
# so now that we have our tasks lets create our crew which is pretty straigth forwardcrew = Crew(
crew = Crew(
    agents=[sales_rep_agent, lead_sales_rep_agent],
    tasks=[lead_profiling_task.dict(), personalized_outreach_task.dict()],
    verbose=2,
    memory=True
)


In [ ]:
inputs = {
    "lead_name": "co-founder and Executive Director",
    "industry": "The AI Collective",
    "key_decision_maker": "Chappy (Gabriel) Asel",
    "position": "Competitive Bodybuilder",
    "milestone": "Global launch, formal incorporation, scaling to 100+ chapters"
}

result = crew.kickoff(inputs=inputs)

In [ ]:
from IPython.display import Markdown
Markdown(result)